In [ ]:
#collecting information about the different patients in the BTO set

In [1]:
#file management
import os
import pandas as pd
import numpy as np

In [2]:
folder_path = "C:\\Users\\maega"
folder_path2= "UFL Dropbox\\Maegan Cremer\\BTO-data\\datashare_20260605T193301Z-3-001\\datashare_060226"
NSR_file = "UF_NSResponsev2_output.csv"

full_path = os.path.join(folder_path, folder_path2, NSR_file)

In [15]:
#for the methylation signatures
folder_path3 = "methylation_signatures"
pathMethylationFiles = os.path.join(folder_path, folder_path2, folder_path3)

In [4]:
#import the NSR file
df_NSR = pd.read_csv(full_path)


In [5]:
#find list of unique in "patient_id" column
unique_patients = df_NSR["patient_id"].unique()
unique_samples = df_NSR['sample_name'].unique()


In [9]:
#section of df_NSR for each patient
patient_dfs = {}
for patient in unique_patients:
    patient_dfs[patient] = df_NSR[df_NSR["patient_id"] == patient].sort_values(by="collection_number", ascending=True)


In [10]:
patient_dfs['BTO-003']

,sample_name,patient_id,collection_number,collection_date,tms,nsrv2_call
1,NS1035CR0007,BTO-003,1,2023-06-15,NaN,NaN
2,NS1035CR0017,BTO-003,2,2023-08-17,NaN,NaN


In [ ]:
patient= 'BTO-009'
samples = patient_dfs[patient]['sample_name']
temp_dfs = {}
for sample in samples:
    methylation_file = os.path.join(pathMethylationFiles, f"{sample}_methylation_signature.tsv")
    if os.path.exists(methylation_file):
        sample_methyl_df = pd.read_csv(methylation_file, sep="\t")
        sample_methyl_df = sample_methyl_df.set_index('locus_name').T
        temp_dfs[sample] = sample_methyl_df
#contatinate teh temp_dfs for each sample into a single df for the patient

what = pd.concat(temp_dfs.values(), axis=0)



In [75]:
what

locus_name,10_q24.2_1,8_q21.3_1,1_p13.3_1,10_q22.1_1,11_q25_1,12_q24.33_1,15_q21.1_1,15_q26.3_1,16_p13.3_2,16_p12.2_1,...,7_q36.3_22,8_q24.22_12,21_q22.11_7,7_q21.3_29,10_q26.13_9,12_q24.33_12,6_p12.1_21,17_p11.2_20,2_q14.2_22,2_q31.1_39
patient_id,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,...,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009,BTO-009
sample_name,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,...,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4,NS1035CR0057-4
chr,10,8,1,10,11,12,15,15,16,16,...,7,8,21,7,10,12,6,17,2,2
band,q24.2,q21.3,p13.3,q22.1,q25,q24.33,q21.1,q26.3,p13.3,p12.2,...,q36.3,q24.22,q22.11,q21.3,q26.13,q24.33,p12.1,p11.2,q14.2,q31.1
locus_weighted_TMS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.003265,0.0,0.0,...,0.670481,-0.000961,1.063212,-0.000013,-0.004372,-0.001947,-0.0,2.787301,4.906304,0.0


In [60]:
what_T = what.T.drop(columns=['sample_name','patient_id', "chr", 'band'], inplace=False)
pt_time = patient_dfs[patient]['collection_date']
#update the date column to be the collection date from the patient_dfs
what2= what_T.T
what2['date'] = pt_time.values
what2 = what2.set_index('date')

In [ ]:
#collect the methylation signature files for each patient
#transform the data such that column names are the different locus names and the rows correspond to teh sample names, and the values are the methylation levels for each locus in each sample
methylation_dfs = {}
for patient in unique_patients:
    samples = patient_dfs[patient]['sample_name']
    temp_dfs = {}
    for sample in samples:
        methylation_file = os.path.join(pathMethylationFiles, f"{sample}_methylation_signature.tsv")
        if os.path.exists(methylation_file):
            sample_methyl_df = pd.read_csv(methylation_file, sep="\t")
            sample_methyl_df = sample_methyl_df.set_index('locus_name').T
            sample_methyl_df.index.name = 'sample_name'
            temp_dfs[sample] = sample_methyl_df
    #contatinate teh temp_dfs for each sample into a single df for the patient
    if temp_dfs:  # Check if temp_dfs is not empty
        concatdf = pd.concat(temp_dfs.values(), axis=0)
        #swap the sample_name for the date from the patient_dfs
        concatdf = concatdf.replace({'sample_name': patient_dfs[patient]['collection_date'].to_dict()})
        concatdf_T = concatdf.T.drop(columns=['sample_name', 'patient_id', "chr", 'band'], inplace=False)
        pt_time = patient_dfs[patient][patient_dfs[patient]['sample_name'] == sample]['collection_date'] #need all dates though not just the last date
        concatdf_2 = concatdf_T.T
        concatdf_2['date'] = pt_time.values
        concatdf_2 = concatdf_2.set_index('date')
        methylation_dfs[patient] = concatdf_2

    

ValueError: Length of values (1) does not match length of index (3)

In [83]:
patient

'BTO-007'

In [84]:
concatdf_2

locus_name,10_q24.2_1,8_q21.3_1,1_p13.3_1,10_q22.1_1,11_q25_1,12_q24.33_1,15_q21.1_1,15_q26.3_1,16_p13.3_2,16_p12.2_1,...,7_q36.3_22,8_q24.22_12,21_q22.11_7,7_q21.3_29,10_q26.13_9,12_q24.33_12,6_p12.1_21,17_p11.2_20,2_q14.2_22,2_q31.1_39
sample_name,,,,,,,,,,,,,,,,,,,,,
locus_weighted_TMS,0.118923,7.261357,0.0,0.0,0.0,6.285144,0.0,0.0,-0.003701,0.371284,...,6.051951,3.596806,0.003356,10.289315,12.248215,21.522951,5.09483,0.103686,0.177697,0.674727
locus_weighted_TMS,0.0,0.0,0.0,0.0,0.0,0.627112,0.0,0.0,0.0,0.0,...,0.128472,-0.003349,0.000918,4.067558,0.389249,1.214456,0.39284,0.120332,0.098779,0.242803
locus_weighted_TMS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.19146,0.0,...,1.04417,0.0,0.09668,0.823399,0.555864,-0.001066,1.742115,-0.000573,1.031327,-0.002949


In [80]:
temp_dfs.keys()

dict_keys(['NS1035CR0057'])

In [81]:
patient_dfs[patient][patient_dfs[patient]['sample_name'] == sample]['collection_date']

25    2023-12-12
Name: collection_date, dtype: object

In [85]:
pt_time

10    2023-08-08
Name: collection_date, dtype: object

In [64]:
patient_dfs['BTO-009']

,sample_name,patient_id,collection_number,collection_date,tms,nsrv2_call
21,NS1035CR0002,BTO-009,1,2023-06-05,NaN,NaN
22,NS1035CR0044,BTO-009,2,2023-07-13,NaN,NaN
23,NS1035CR0023,BTO-009,3,2023-08-28,NaN,NaN
24,NS1035CR0057,BTO-009,4,2023-09-26,332.85287,baseline
25,NS1035CR0144,BTO-009,5,2023-12-12,NaN,NaN
